# Avatar Falante PT-BR - Colab (GPU T4)

1. `Ambiente > Alterar tipo de ambiente de execucao > GPU T4`
2. Rode as celulas em ordem. A ultima gera o link publico do Gradio (`share=True`).

> Instalacao do SadTalker no Colab depende das versoes de Python/PyTorch do momento; se alguma celula falhar, veja a secao *Problemas conhecidos* do README.

In [ ]:
!nvidia-smi
import torch; print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Ative a GPU T4 no menu de ambiente de execucao.'

In [ ]:
# Envie o arquivo meu-avatar.zip (zip da pasta do projeto, sem .venv)
from google.colab import files
import zipfile, os
up = files.upload()
nome = next(iter(up))
zipfile.ZipFile(nome).extractall('/content/meu-avatar')
%cd /content/meu-avatar
!ls

In [ ]:
!apt-get -qq install -y ffmpeg
!pip -q install gradio edge-tts 'rembg[gpu]' onnxruntime-gpu imageio-ffmpeg diffusers transformers accelerate huggingface_hub

In [ ]:
# SadTalker + GFPGAN (checkpoints ~ alguns GB)
!git clone --depth 1 https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip -q install face_alignment imageio imageio-ffmpeg librosa resampy pydub kornia yacs scikit-image basicsr facexlib gfpgan safetensors av 'numpy<2'
!bash scripts/download_models.sh
%cd ..

In [ ]:
# Correcoes comuns de compatibilidade do SadTalker com numpy/torchvision recentes
import re, pathlib
for f in pathlib.Path('SadTalker').rglob('*.py'):
    t = f.read_text()
    n = t.replace('np.float,', 'float,').replace('np.float)', 'float)').replace('np.int)', 'int)')
    if n != t: f.write_text(n)
# basicsr importa torchvision.transforms.functional_tensor (removido nas versoes novas)
import site, glob
for f in glob.glob(site.getsitepackages()[0] + '/basicsr/data/degradations.py'):
    t = open(f).read().replace('torchvision.transforms.functional_tensor', 'torchvision.transforms.functional')
    open(f, 'w').write(t)

In [ ]:
# Sobe o app com link publico. O SadTalker e detectado automaticamente.
import os
os.environ['SADTALKER_DIR'] = '/content/meu-avatar/SadTalker'
from app import construir
demo = construir()
demo.queue().launch(share=True, debug=True)

### Usar o Colab como motor de animacao do app local
Copie a URL publica (`https://xxxx.gradio.live`) e, na maquina local:
```bash
REMOTE_ANIMADOR_URL=https://xxxx.gradio.live .venv/bin/python app.py
```